# Part 1 : Snowflake 基礎

## このノートブックでやること
1. 仮想ウェアハウスの作成と設定
2. ステージからのデータ取り込み

In [ ]:
-- コンテキスト設定
USE ROLE ACCOUNTADMIN;
USE DATABASE GLACIERSTYLE_DB;
USE SCHEMA EC_ANALYTICS_SCHEMA;

-- ハンズオン中はリザルトキャッシュを無効化（WH の挙動を正しく体験するため）
ALTER SESSION SET USE_CACHED_RESULT = FALSE;


## 1. 仮想ウェアハウスの作成と設定

仮想ウェアハウスは、クエリやデータ処理を実行するための**コンピューティングリソース**です。
Snowflake ではストレージとコンピューティングが分離されているため、ウェアハウスを自由にスケールアップ/ダウンできます。

![WH Overview](images/part1/01_warehouse_overview.png)

### 主なパラメータ

![WH Params](images/part1/02_warehouse_params.png)

| パラメータ | 説明 | デフォルト |
|---|---|---|
| `WAREHOUSE_SIZE` | コンピューティングサイズ（X-Small 〜 X-Large 等） | XSmall |
| `AUTO_SUSPEND` | 非アクティブ後に自動停止するまでの秒数 | 600秒 |
| `AUTO_RESUME` | クエリ送信時に自動で再開するか | TRUE |
| `INITIALLY_SUSPENDED` | 作成直後に停止状態で開始するか | TRUE |

In [ ]:
-- 既存のウェアハウスを確認
SHOW WAREHOUSES;


In [ ]:
-- ウェアハウスを作成（AUTO_RESUME = OFF）
CREATE OR REPLACE WAREHOUSE my_wh
    WAREHOUSE_TYPE      = 'standard'
    WAREHOUSE_SIZE      = 'xsmall'
    AUTO_SUSPEND        = 60
    INITIALLY_SUSPENDED = true -- 作成直後は停止状態
    AUTO_RESUME         = false; -- 自動再開しない


In [ ]:
-- このワークシートで使用するウェアハウスを指定
USE WAREHOUSE my_wh;

-- 現在の状態を確認
SHOW WAREHOUSES LIKE 'MY_WH';


In [ ]:
-- 停止中のままクエリを投げてみる → エラーになるはず
SELECT * FROM dim_customers LIMIT 5;


エラーになりましたね。AUTO_RESUME が OFF なので、停止中のウェアハウスではクエリを実行できません。

じゃあ AUTO_RESUME を ON にしてもう一回やってみましょう 👇

In [ ]:
-- COMPUTE_WH に切り替えてから設定変更
USE WAREHOUSE COMPUTE_WH;
ALTER WAREHOUSE my_wh SET AUTO_RESUME = TRUE;


In [ ]:
-- my_wh に戻してクエリを投げる
USE WAREHOUSE my_wh;
SELECT * FROM dim_customers LIMIT 5;


### スケールアップ / スケールダウン

SQL コマンド 1 行でウェアハウスサイズを即座に変更できます。
重いクエリを実行するときだけスケールアップして、終わったらスケールダウン、という使い方ができます。

> **ポイント:** サイズを 1 段上げるとコンピューティングリソースが 2 倍になります。XSmall → Large で 8 倍。

In [ ]:
-- スケールアップ: XSmall → XLarge
ALTER WAREHOUSE my_wh SET WAREHOUSE_SIZE = 'XLarge';


In [ ]:
-- XLarge で集計クエリを実行
SELECT
    c.membership_tier,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.total_amount)        AS total_sales
FROM fact_orders o
JOIN dim_customers c ON o.customer_id = c.customer_id
GROUP BY c.membership_tier
ORDER BY total_sales DESC;


In [ ]:
-- 作業が終わったらスケールダウン
ALTER WAREHOUSE my_wh SET WAREHOUSE_SIZE = 'XSmall';


---
## 2. ステージからのデータ取り込み

Snowflake の**ステージ**は、データファイルが置かれた場所を参照するオブジェクトです。
`COPY INTO` コマンドを使ってステージからテーブルへデータをロードします。

### ステージの種類

![Stage Types](images/part1/02_external_stage_1.png)

| 種類 | 説明 |
|---|---|
| **内部ステージ** | Snowflake 管理のストレージ。Snowsight からファイルをアップロードできる |
| **外部ステージ** | S3 / Azure Blob / GCS などクラウドストレージを参照 |


![COPY INTO Flow](images/part1/03_external_stage_2.png)

In [ ]:
-- DATA_STAGE の確認（setup.sql で作成済み）
-- ※ ステージ自体は setup.sql で作成してあります
DESCRIBE STAGE DATA_STAGE;


In [ ]:
-- ステージ内のファイル一覧を確認（アップロード済みのファイルが表示される）
LIST @DATA_STAGE;


In [ ]:
-- ロード先テーブルを作成
-- ※ setup.sql で既に作成・データロード済みですが、ここでは手順を体験します
CREATE OR REPLACE TABLE dim_customers (
    customer_id       VARCHAR PRIMARY KEY,
    email             VARCHAR,
    phone             VARCHAR,
    last_name         VARCHAR,
    first_name        VARCHAR,
    gender            VARCHAR,
    birth_date        DATE,
    postal_code       VARCHAR,
    prefecture        VARCHAR,
    city              VARCHAR,
    address           VARCHAR,
    registration_date DATE,
    membership_tier   VARCHAR,
    total_orders      INTEGER,
    total_spent       DECIMAL(12,2),
    last_order_date   DATE,
    email_opt_in      BOOLEAN,
    app_installed     BOOLEAN
);


In [ ]:
-- ステージからテーブルへデータをロード
COPY INTO dim_customers
FROM @DATA_STAGE/data/customers.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"');


In [ ]:
-- ロード結果を確認
SELECT * FROM dim_customers LIMIT 20;


---
## クリーンアップ

作成したオブジェクトを削除する場合は以下を実行してください。

In [ ]:
USE ROLE ACCOUNTADMIN;
DROP WAREHOUSE IF EXISTS my_wh;
-- ※ dim_customers は Part 2 でも使うため削除しない
